BERT Crisis Classifier — 3-class scheme
0 = no_crisis, 1 = implicit_crisis, 2 = explicit_crisis (collapsed from severity 2-6)

Run in Google Colab: Runtime > Change runtime type > GPU (T4 is fine)

Changes vs. your last run (the one in the report):
  - Labels collapsed 0 / 1 / 2-6 -> 0 / 1 / 2
  - Class-weighted loss enabled (was explicitly OFF before) — this is the main fix
    for the model never predicting the minority "implicit" class
  - More epochs (8, with early stopping) instead of a fixed 3
  - Best checkpoint selected by macro F1, same as before

## 0. SETUP

In [1]:
# !pip install -q transformers datasets scikit-learn torch --upgrade

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from torch.utils.data import Dataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 1. UPLOAD / LOAD DATA

In [2]:
# Option A: upload directly in Colab
from google.colab import files
# uploaded = files.upload()  # choose human_n_llm_labeled_rSuicidewatch_posts.csv

# Option B: mount Drive and point CSV_PATH at the file there
# from google.colab import drive
# drive.mount('/content/drive')
uploaded = files.upload()
CSV_PATH = "merged_severity_dataset_not_keyword_keyword.csv"  # adjust path if needed

df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["content", "severity"]).reset_index(drop=True)


def map_severity_to_3class(sev):
    sev = int(sev)
    if sev == 0:
        return 0  # no_crisis
    elif sev == 1:
        return 1  # implicit_crisis
    else:
        return 2  # explicit_crisis (severity 2-6 collapsed)


df["label"] = df["severity"].apply(map_severity_to_3class)

print("Label distribution after collapsing to 3 classes:")
print(df["label"].value_counts().sort_index())

texts = df["content"].astype(str).tolist()
labels = df["label"].tolist()

Saving merged_severity_dataset_not_keyword_keyword.csv to merged_severity_dataset_not_keyword_keyword.csv
Label distribution after collapsing to 3 classes:
label
0    351
1    324
2    619
Name: count, dtype: int64


## 2. TRAIN / VAL / TEST SPLIT (stratified 70/15/15)

In [3]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.30, random_state=SEED, stratify=labels
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.50, random_state=SEED, stratify=temp_labels
)

print(f"Train: {len(train_texts)}  Val: {len(val_texts)}  Test: {len(test_texts)}")
print("Train label counts:", pd.Series(train_labels).value_counts().sort_index().to_dict())

Train: 905  Val: 194  Test: 195
Train label counts: {0: 245, 1: 227, 2: 433}


## 3. TOKENIZATION / DATASET

In [4]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)


class CrisisDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length", max_length=max_len
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


train_dataset = CrisisDataset(train_texts, train_labels, tokenizer)
val_dataset = CrisisDataset(val_texts, val_labels, tokenizer)
test_dataset = CrisisDataset(test_texts, test_labels, tokenizer)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## 4. CLASS WEIGHTS — this is the key fix for the imbalance problem

In [5]:
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=np.array(train_labels),
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(device)
print("Class weights (no_crisis, implicit_crisis, explicit_crisis):", class_weights_np)

Class weights (no_crisis, implicit_crisis, explicit_crisis): [1.23129252 1.32892805 0.69668976]


## 5. MODEL + WEIGHTED LOSS TRAINER

In [6]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. METRICS

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": acc,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }

## 7. TRAINING ARGS

In [8]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=int(0.1 * (len(train_dataset) / 16) * 8),  # ~10% of total steps
    weight_decay=0.01,
    eval_strategy="epoch",       # older transformers: use evaluation_strategy="epoch"
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

## 8. TRAIN

In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,1.091787,1.004855,0.592784,0.649307,0.508005,0.509426
2,0.892844,0.728373,0.664948,0.699848,0.688007,0.663795
3,0.605240,0.574627,0.788660,0.786449,0.799030,0.782995
4,0.355636,0.503786,0.850515,0.850433,0.829994,0.838294
5,0.204334,0.590458,0.840206,0.829910,0.832906,0.829477
6,0.078438,0.663168,0.835052,0.823758,0.823912,0.820942
7,0.038124,0.659282,0.855670,0.847660,0.845709,0.846326
8,0.016347,0.686509,0.850515,0.841478,0.839420,0.839770


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=456, training_loss=0.36243210473963827, metrics={'train_runtime': 421.3988, 'train_samples_per_second': 17.181, 'train_steps_per_second': 1.082, 'total_flos': 952470572175360.0, 'train_loss': 0.36243210473963827, 'epoch': 8.0})

## 9. EVALUATE ON TEST SET (single pass, best checkpoint)

In [10]:
test_results = trainer.predict(test_dataset)
test_preds = np.argmax(test_results.predictions, axis=1)
test_true = test_results.label_ids

label_names = ["no_crisis", "implicit_crisis", "explicit_crisis"]

print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)
print("Accuracy:", accuracy_score(test_true, test_preds))
print()
print(
    classification_report(
        test_true, test_preds, target_names=label_names, digits=4, zero_division=0
    )
)
print("Confusion matrix (rows = true, cols = predicted):")
print(
    pd.DataFrame(
        confusion_matrix(test_true, test_preds),
        index=[f"true_{n}" for n in label_names],
        columns=[f"pred_{n}" for n in label_names],
    )
)

FINAL TEST RESULTS
Accuracy: 0.7897435897435897

                 precision    recall  f1-score   support

      no_crisis     0.7708    0.6981    0.7327        53
implicit_crisis     0.7556    0.6939    0.7234        49
explicit_crisis     0.8137    0.8925    0.8513        93

       accuracy                         0.7897       195
      macro avg     0.7800    0.7615    0.7691       195
   weighted avg     0.7875    0.7897    0.7869       195

Confusion matrix (rows = true, cols = predicted):
                      pred_no_crisis  pred_implicit_crisis  \
true_no_crisis                    37                     7   
true_implicit_crisis               5                    34   
true_explicit_crisis               6                     4   

                      pred_explicit_crisis  
true_no_crisis                           9  
true_implicit_crisis                    10  
true_explicit_crisis                    83  


## 10. SAVE MODEL

In [11]:
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")
print("Model saved to ./best_model")

# Optional: zip and download the trained model from Colab
# !zip -r best_model.zip best_model
# from google.colab import files
# files.download("best_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./best_model


In [12]:
!zip -r best_model.zip ./best_model

  adding: best_model/ (stored 0%)
  adding: best_model/model.safetensors (deflated 7%)
  adding: best_model/tokenizer_config.json (deflated 43%)
  adding: best_model/training_args.bin (deflated 53%)
  adding: best_model/tokenizer.json (deflated 71%)
  adding: best_model/config.json (deflated 54%)


In [13]:
from google.colab import files

files.download("best_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>